# Data Engineering dan Document Processing BANASPATI

Notebook ini mendokumentasikan ekstraksi multimodal, metadata sumber, eksperimen chunking, embedding, dan vector database. Semua output disimpan sebagai artefak agar dapat diaudit dan digunakan anggota tim lain.

## 1. Setup

Model embedding multilingual dipilih karena dokumen dan pertanyaan dominan berbahasa Indonesia. ChromaDB dipakai untuk menyimpan embedding sekaligus metadata sumber.

In [ ]:
%pip install -q -r ../requirements.txt

## 2. Ekstraksi Multimodal

PDF diekstrak per halaman; DOCX per section. Tabel disimpan sebagai teks berpemisah `|`, sedangkan gambar dicatat dan dapat diproses OCR. Metadata halaman/section dipertahankan untuk sitasi.

In [ ]:
!python ../scripts/data_pipeline.py prepare --input ../data/database --output ../artifacts

import json, pandas as pd
report = json.load(open('../artifacts/chunking_experiment.json'))
pd.DataFrame(report)

## 3. Interpretasi Eksperimen Chunking

`fixed_500_100` adalah baseline. Recursive chunking menjaga batas kalimat sehingga konteks lebih koheren. Konfigurasi `recursive_1000_150` digunakan sebagai kandidat awal karena memberi konteks yang cukup tanpa membuat chunk terlalu lebar. Keputusan final harus divalidasi memakai retrieval recall/precision pada pertanyaan evaluasi.

## 4. Embedding dan Vector Database

Indeks berikut menggunakan cosine similarity. Setiap hasil retrieval tetap membawa nama file, halaman/section, dan jenis konten.

In [ ]:
!python ../scripts/data_pipeline.py index --chunks ../artifacts/chunks_recursive_1000_150.jsonl --persist-dir ../artifacts/chroma

In [ ]:
!python ../scripts/data_pipeline.py query 'Apa aturan akademik yang berlaku?' --persist-dir ../artifacts/chroma --top-k 5

## 5. Limitasi

Struktur tabel kompleks dapat berubah saat dikonversi ke teks. OCR bergantung kualitas scan. Gambar non-teks baru dicatat keberadaannya dan memerlukan image captioning untuk retrieval semantik. Pemilihan chunk terbaik juga belum valid tanpa evaluasi terhadap pertanyaan dan ground truth.